In [39]:
import json
import re
import os
from pathlib import Path
import pandas as pd

RESULT_FOLDER_PATH = "/Users/lockewang/FIG/WebDomainRandomizer/results_baseline"

# Load all data from uitars_predictions and uitars_metrics

In [40]:
import pandas as pd

def load_metrics_dict(metrics_file):
    """Load all metrics from uitars_metrics.jsonl into a dictionary keyed by (episode, step_index)"""
    metrics_dict = {}
    if not metrics_file.exists():
        return metrics_dict
    
    try:
        with open(metrics_file, 'r') as f:
            for line in f:
                data = json.loads(line)
                episode = data.get('episode')
                step_index = data.get('step_index')
                if episode is not None and step_index is not None:
                    key = (episode, step_index)
                    metrics_dict[key] = data.get('metrics', {})
    except Exception as e:
        print(f"Warning: Error reading metrics file {metrics_file}: {e}")
    
    return metrics_dict

def load_data_from_results_folder(result_folder_path):
    """Load all data from uitars_predictions jsonl files and match with metrics"""
    rows = []
    result_path = Path(result_folder_path)
    
    for model_config_folder in result_path.iterdir():
        if not model_config_folder.is_dir():
            continue
        if model_config_folder.name == 'locke_qwen_thought_relational_query_2':
            continue

        print(f"Processing model config folder: {model_config_folder.name}")
        
        for run_folder in model_config_folder.iterdir():
            if not run_folder.is_dir():
                continue
            
            if not run_folder.name.startswith('run'):
                continue

            model = model_config_folder.name.split('_')[1]
            reasoning_type = "no_reasoning" if "NO" in model_config_folder.name else "reasoning"
            query_type = "direct_query" if "direct_query" in model_config_folder.name else "relational_query"
            test_split = run_folder.name.split('_')[4]
            variant = "_".join(run_folder.name.split('_')[5:])
            
            predictions_folder = run_folder / "uitars_predictions"
            metrics_file = run_folder / "uitars_metrics.jsonl"
            
            if not predictions_folder.exists():
                continue
            
            # Load all metrics for this run_folder into a dictionary
            metrics_dict = load_metrics_dict(metrics_file)
            
            # Process each task_id jsonl file in uitars_predictions
            for prediction_file in predictions_folder.glob("*.jsonl"):
                task_id = prediction_file.stem  # Get task_id from filename (without .jsonl extension)
                
                try:
                    with open(prediction_file, 'r') as f:
                        for line in f:
                            data = json.loads(line)
                            
                            episode = data.get('episode', task_id)
                            step_index = data['step_index']
                            instruction = data['instruction']
                            prediction = data['prediction']

                            # Get ground_truth_bbox from metadata
                            metadata = data.get('metadata', {})
                            gt_bbox = metadata.get('bounding_box')

                            # Get metrics from metrics_dict
                            step_metrics = metrics_dict.get((episode, step_index), {})
                            
                            row = {
                                'model': model,
                                'reasoning_type': reasoning_type,
                                'query_type': query_type,
                                'test_split': test_split,
                                'variant': variant,
                                'task_id': episode,
                                'step_index': step_index,
                                'instruction': instruction,
                                'gt_bbox': gt_bbox,
                                'prediction': prediction,
                                'action_str_em': step_metrics.get('action_str_em'),
                                'hit_box_accuracy': step_metrics.get('hit_box_accuracy'),
                                'bbox_center_mse': step_metrics.get('bbox_center_mse')
                            }
                            
                            rows.append(row)
                            
                except Exception as e:
                    print(f"Warning: Error reading prediction file {prediction_file}: {e}")
                    continue
    
    return pd.DataFrame(rows)

# Load all data
df = load_data_from_results_folder(RESULT_FOLDER_PATH)

print(f"Total rows loaded: {len(df)}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()


Processing model config folder: locke_qwen_NO_thought_relational_query
Processing model config folder: locke_qwen_thought_relational_query
Processing model config folder: locke_gta1_NO_thought_direct_query
Processing model config folder: locke_uitars_NO_thought_relational_query
Processing model config folder: locke_uitars_thought_direct_query
Processing model config folder: locke_uitars_thought_relational_query
Processing model config folder: locke_uitars_NO_thought_direct_query
Processing model config folder: locke_qwen_thought_direct_query
Processing model config folder: locke_gta1_NO_thought_relational_query
Processing model config folder: locke_qwen_NO_thought_direct_query
Total rows loaded: 11835

Columns: ['model', 'reasoning_type', 'query_type', 'test_split', 'variant', 'task_id', 'step_index', 'instruction', 'gt_bbox', 'prediction', 'action_str_em', 'hit_box_accuracy', 'bbox_center_mse']

First few rows:


,model,reasoning_type,query_type,test_split,variant,task_id,step_index,instruction,gt_bbox,prediction,action_str_em,hit_box_accuracy,bbox_center_mse
0,qwen,no_reasoning,relational_query,task,text_shrink,8dcf6423-262a-439b-9ee7-279a920468fa,0,Click on the button to the left of 'EXPERIENCE',"[370, 61, 111.265625, 55]","Action: click(start_box='(553,86)')",1.0,0.0,8114.325226
1,qwen,no_reasoning,relational_query,task,text_shrink,e6f6e6c8-f1e6-42bb-a3af-696ed8de571b,0,Click on the link above 'Flight status',"[607.046875, 174.390625, 65.390625, 12]","Action: click(start_box='(640,211)')",1.0,0.0,468.500153
2,qwen,no_reasoning,relational_query,task,text_shrink,400c291f-6a0c-46fb-874e-d5c174fdedfc,0,Click on the button below 'Infant +' button,"[895.5, 533.5, 32.5, 13]","Action: click(start_box='(834,500)')",1.0,0.0,3822.531250
3,qwen,no_reasoning,relational_query,task,text_shrink,5b888855-b921-4c61-8f79-73902ee0eafa,0,Type the textbox above '* PIN Number',"[448, 461.953125, 380, 36.890625]","Action: click(start_box='(641,563)')",0.0,0.0,3416.009064
4,qwen,no_reasoning,relational_query,task,text_shrink,9ceab2a3-7919-4f15-871a-21638fd93b24,0,"Select '11:00 AM' in the select to the left of 'Drop-off date Mon, Apr 3'","[1036.40625, 202, 112, 62]","Action: click(start_box='(1356,247)')",0.0,0.0,34838.832520


# Fix nan instruction

In [41]:
nan_instruction = df['instruction'].apply(lambda x: str(x).split(' ')[0].lower() if pd.notna(x) else '') == 'nan'
print(f"Number of rows with 'nan' instruction: {nan_instruction.sum()}")
print(f"\nAll rows with 'nan' instruction:")
display(df[nan_instruction].head(5))


nan_gt_bbox = df['gt_bbox'].apply(lambda x: x == 'nan')
print(f"Number of rows with 'nan' gt_bbox: {nan_gt_bbox.sum()}")
print(f"\nAll rows with 'nan' gt_bbox:")
display(df[nan_gt_bbox].head(5))


Number of rows with 'nan' instruction: 20

All rows with 'nan' instruction:


,model,reasoning_type,query_type,test_split,variant,task_id,step_index,instruction,gt_bbox,prediction,action_str_em,hit_box_accuracy,bbox_center_mse
508,qwen,no_reasoning,relational_query,website,style,f5b9e5b9-1992-4444-b389-680912e50fe6,0,nan,"[31, 220.09375, 440.5, 48]","Action: click(start_box='(67,423)')",0.0,0.0,32977.754395
689,qwen,no_reasoning,relational_query,website,text_shrink,f5b9e5b9-1992-4444-b389-680912e50fe6,0,nan,"[31, 213.09375, 440.5, 48]","Action: click(start_box='(253,240)')",0.0,1.0,5.754395
741,qwen,no_reasoning,relational_query,website,precision,f5b9e5b9-1992-4444-b389-680912e50fe6,0,nan,"[281.6953125, 149.16561889648438, 322.3499755859375, 33.600006103515625]","Action: click(start_box='(445,340)')",0.0,0.0,15146.250182
792,qwen,no_reasoning,relational_query,website,original,f5b9e5b9-1992-4444-b389-680912e50fe6,0,nan,"[31, 213.09375, 440.5, 48]","Action: click(start_box='(253,240)')",0.0,1.0,5.754395
1692,qwen,reasoning,relational_query,website,style,f5b9e5b9-1992-4444-b389-680912e50fe6,0,nan,"[31, 220.09375, 440.5, 48]","Thought: The goal is to explore job listings within a specific area. The current screen shows job results with a location filter set to ""20001"" and ""25 miles."" To view more jobs, I should scroll down to see additional listings.\n\nAction: scroll(start_box='Box(498,167,1919,1087)')",NaN,NaN,NaN


Number of rows with 'nan' gt_bbox: 0

All rows with 'nan' gt_bbox:


,model,reasoning_type,query_type,test_split,variant,task_id,step_index,instruction,gt_bbox,prediction,action_str_em,hit_box_accuracy,bbox_center_mse


In [42]:
test_data_folder = "/Users/lockewang/FIG/WebDomainRandomizer/test_splits"
run_folders = list(Path(test_data_folder).iterdir())

def find_instruction_with_nan(row):
    test_split = row['test_split']
    variant = row['variant']
    task_id = row['task_id']
    step_index = row['step_index']

    for run_folder in run_folders:
        if test_split in run_folder.name and variant in run_folder.name:
            json_file = run_folder / task_id / "trajectory.json"
            with open(json_file, 'r') as f:
                json_data = json.load(f)
                return json_data[step_index]['step_instruction']

    raise ValueError(f"No instruction found for task_id: {task_id}, step_index: {step_index}")

df['instruction'] = df.apply(lambda x: find_instruction_with_nan(x) if x['instruction'] == 'nan' else x['instruction'], axis=1)


In [47]:
nan_instruction = df['instruction'].apply(lambda x: str(x).split(' ')[0].lower() if pd.notna(x) else '') == 'nan'

len(df[nan_instruction])
df[nan_instruction].head(5)

,model,reasoning_type,query_type,test_split,variant,task_id,step_index,instruction,gt_bbox,prediction,action_str_em,hit_box_accuracy,bbox_center_mse


# For each model_config, check how many types of predictions are there

In [3]:
def categorize_prediction_pattern(text):
    """Categorize prediction text into pattern types"""
    if pd.isna(text):
        return 'missing'
    
    text_str = str(text)
    
    types = []

    # Check if contains 'Thought' (case-insensitive)
    if 'Thought' in text_str or 'thought' in text_str:
        types.append('has_thought')
    else:
        types.append('no_thought')
    
    # Check if contains 'tool_call' (case-insensitive)
    if 'tool_call' in text_str or 'toolCall' in text_str or 'tool-call' in text_str:
        types.append('contains_tool_call')
    
    # Check if has exactly 4 numbers (could be bbox format like [x1, y1, x2, y2] or (x1, y1, x2, y2))
    numbers = re.findall(r'\d+', text_str)
    if len(numbers) == 4:
        types.append('4_numbers')

    # Extract all actions after "Action:" - handle multiple actions separated by commas
    action_section_match = re.search(r'Action:\s*(.+?)(?:\n|$)', text_str, re.IGNORECASE | re.DOTALL)
    if action_section_match:
        action_section = action_section_match.group(1).strip()
        
        # Parse multiple actions (they're separated by commas, but we need to be careful with nested parentheses)
        # Simple approach: find all action calls like action_name(...)
        action_pattern = r'\b(click|scroll|type|select|wait|finished)\s*\('
        found_actions = re.findall(action_pattern, action_section, re.IGNORECASE)
        
        # Add all found action types
        for action in found_actions:
            action_lower = action.lower()
            if action_lower not in [t.lower() for t in types]:
                types.append(action_lower)
    
    # Everything else is unexpected
    if len(types) == 0:
        types.append('other_unexpected')

    return ' + '.join(types)

In [4]:
pd.set_option('display.max_rows', None)

df['prediction_type'] = df['prediction'].apply(categorize_prediction_pattern)

other_unexpected_count = df[df['prediction_type'] == 'other_unexpected'].sum()
other_unexpected_count


model                 0
reasoning_type        0
query_type            0
test_split            0
variant               0
task_id               0
step_index            0
instruction           0
gt_bbox               0
prediction            0
action_str_em       0.0
hit_box_accuracy    0.0
bbox_center_mse     0.0
prediction_type       0
dtype: object

In [5]:
# filter out rows with expected prediction types
# reasoning_type is reasoning, prediction_type should only have contains_thought and other actions like select, scroll, type, etc.
# reasoning_type is no_reasoning, prediction_type should only have simple_click and other actions like select, scroll, type, etc.

expected_prediction_filter = (
    ((df['reasoning_type'] == 'reasoning') & 
     (df['prediction_type'].apply(lambda x: all(action in ['has_thought', 'click', 'scroll', 'type', 'select', 'wait', 'finished'] for action in x.split(' + '))))) |
    ((df['reasoning_type'] == 'no_reasoning') & 
     (df['prediction_type'].apply(lambda x: all(action in ['no_thought', 'click', 'scroll', 'type', 'select', 'wait', 'finished'] for action in x.split(' + ')))))
)
filtered_df = df[~expected_prediction_filter]
expected_df = df[expected_prediction_filter]

print(f'Filtered out {len(df) - len(filtered_df)} / {len(df)} rows')

Filtered out 10691 / 11835 rows


## Unexpected predictions
Only Qwen has all 3 malformed predictions, and UITARS has some 4D coordinates cases
1. has <tool_call>...scroll....pixel: -500... or <tool_call>...type...content: PUNT...
2. has 4D coordinates instead 2D
3. had 2 actions, the only case is ["scroll", "wait"]

Side note, 
- our original "predicted_actions" field sometimes has 2 actions like 'click', 'type'. But this is from our parsing logic. The 'prediction' string has 'type' in its 'Thought' but only has 'click' in 'Action'
- saved 'prediction' is not raw prediction, it's formatted prediction. The regex patterns used miss coordinates formatted with (x, y) instead of [x, y] for Qwen. However, we can count (x, y) just as malformed like other cases (e.g., 'scroll... pixel: -500')


In [6]:
for model in filtered_df['model'].unique():
    result = filtered_df[filtered_df['model'] == model].groupby(['model', 'reasoning_type', 'prediction_type'])['prediction'].count()
    styled = (result.reset_index(name='count')
              .style
              .background_gradient(subset='count')
              .format({'count': '{:,}'}))
    print("="*80)
    print(model)
    display(styled)
    print("="*80)


qwen


,model,reasoning_type,prediction_type,count
0,qwen,no_reasoning,no_thought + contains_tool_call,3
1,qwen,reasoning,has_thought + 4_numbers + click,896
2,qwen,reasoning,has_thought + 4_numbers + scroll,112
3,qwen,reasoning,has_thought + 4_numbers + scroll + wait,2
4,qwen,reasoning,has_thought + 4_numbers + wait,1


uitars


,model,reasoning_type,prediction_type,count
0,uitars,reasoning,has_thought + 4_numbers + click,122
1,uitars,reasoning,has_thought + 4_numbers + scroll,4
2,uitars,reasoning,has_thought + 4_numbers + wait,4


In [7]:
for model in filtered_df['model'].unique():
    result = filtered_df[filtered_df['model'] == model].groupby(['model', 'reasoning_type', 'query_type', 'prediction_type'])['prediction'].count()
    styled = (result.reset_index(name='count')
              .style
              .background_gradient(subset='count')
              .format({'count': '{:,}'}))
    print("="*80)
    print(model)
    display(styled)
    print("="*80)

qwen


,model,reasoning_type,query_type,prediction_type,count
0,qwen,no_reasoning,direct_query,no_thought + contains_tool_call,1
1,qwen,no_reasoning,relational_query,no_thought + contains_tool_call,2
2,qwen,reasoning,direct_query,has_thought + 4_numbers + click,435
3,qwen,reasoning,direct_query,has_thought + 4_numbers + scroll,70
4,qwen,reasoning,direct_query,has_thought + 4_numbers + scroll + wait,1
5,qwen,reasoning,direct_query,has_thought + 4_numbers + wait,1
6,qwen,reasoning,relational_query,has_thought + 4_numbers + click,461
7,qwen,reasoning,relational_query,has_thought + 4_numbers + scroll,42
8,qwen,reasoning,relational_query,has_thought + 4_numbers + scroll + wait,1


uitars


,model,reasoning_type,query_type,prediction_type,count
0,uitars,reasoning,direct_query,has_thought + 4_numbers + click,62
1,uitars,reasoning,direct_query,has_thought + 4_numbers + scroll,3
2,uitars,reasoning,direct_query,has_thought + 4_numbers + wait,3
3,uitars,reasoning,relational_query,has_thought + 4_numbers + click,60
4,uitars,reasoning,relational_query,has_thought + 4_numbers + scroll,1
5,uitars,reasoning,relational_query,has_thought + 4_numbers + wait,1


## Expected predictions

In [8]:
for model in expected_df['model'].unique():
    result = expected_df[expected_df['model'] == model].groupby(['model', 'reasoning_type', 'query_type', 'prediction_type'])['prediction'].count()
    styled = (result.reset_index(name='count')
              .style
              .background_gradient(subset='count')
              .format({'count': '{:,}'}))
    print("="*80)
    print(model)
    display(styled)
    print("="*80)

qwen


,model,reasoning_type,query_type,prediction_type,count
0,qwen,no_reasoning,direct_query,no_thought + click,"1,183"
1,qwen,no_reasoning,relational_query,no_thought + click,"1,182"
2,qwen,reasoning,direct_query,has_thought + click,492
3,qwen,reasoning,direct_query,has_thought + finished,5
4,qwen,reasoning,direct_query,has_thought + scroll,150
5,qwen,reasoning,direct_query,has_thought + scroll + wait,6
6,qwen,reasoning,direct_query,has_thought + type,23
7,qwen,reasoning,direct_query,has_thought + wait,1
8,qwen,reasoning,relational_query,has_thought + click,559
9,qwen,reasoning,relational_query,has_thought + finished,8


gta1


,model,reasoning_type,query_type,prediction_type,count
0,gta1,no_reasoning,direct_query,no_thought + click,"1,184"
1,gta1,no_reasoning,relational_query,no_thought + click,"1,184"


uitars


,model,reasoning_type,query_type,prediction_type,count
0,uitars,no_reasoning,direct_query,no_thought + click,"1,148"
1,uitars,no_reasoning,direct_query,no_thought + scroll,36
2,uitars,no_reasoning,relational_query,no_thought + click,"1,086"
3,uitars,no_reasoning,relational_query,no_thought + scroll,98
4,uitars,reasoning,direct_query,has_thought + click,983
5,uitars,reasoning,direct_query,has_thought + finished,7
6,uitars,reasoning,direct_query,has_thought + scroll,59
7,uitars,reasoning,direct_query,has_thought + type,2
8,uitars,reasoning,direct_query,has_thought + wait,65
9,uitars,reasoning,relational_query,has_thought + click,954


# Parsing actions & coordinates from saved 'prediction' field in the df from the jsonl files

In [9]:
df.head()

,model,reasoning_type,query_type,test_split,variant,task_id,step_index,instruction,gt_bbox,prediction,action_str_em,hit_box_accuracy,bbox_center_mse,prediction_type
0,qwen,no_reasoning,relational_query,task,text_shrink,8dcf6423-262a-439b-9ee7-279a920468fa,0,Click on the button to the left of 'EXPERIENCE',"[370, 61, 111.265625, 55]","Action: click(start_box='(553,86)')",1.0,0.0,8114.325226,no_thought + click
1,qwen,no_reasoning,relational_query,task,text_shrink,e6f6e6c8-f1e6-42bb-a3af-696ed8de571b,0,Click on the link above 'Flight status',"[607.046875, 174.390625, 65.390625, 12]","Action: click(start_box='(640,211)')",1.0,0.0,468.500153,no_thought + click
2,qwen,no_reasoning,relational_query,task,text_shrink,400c291f-6a0c-46fb-874e-d5c174fdedfc,0,Click on the button below 'Infant +' button,"[895.5, 533.5, 32.5, 13]","Action: click(start_box='(834,500)')",1.0,0.0,3822.531250,no_thought + click
3,qwen,no_reasoning,relational_query,task,text_shrink,5b888855-b921-4c61-8f79-73902ee0eafa,0,Type the textbox above '* PIN Number',"[448, 461.953125, 380, 36.890625]","Action: click(start_box='(641,563)')",0.0,0.0,3416.009064,no_thought + click
4,qwen,no_reasoning,relational_query,task,text_shrink,9ceab2a3-7919-4f15-871a-21638fd93b24,0,Select '11:00 AM' in the select to the left of...,"[1036.40625, 202, 112, 62]","Action: click(start_box='(1356,247)')",0.0,0.0,34838.832520,no_thought + click


In [10]:
# parsing prediction from the df

def parse_actions(prediction_str):
    """
    Extract all actions from a prediction string.
    Handles: "Action: click(...)", JSON in tool_call tags, and malformed cases.
    Returns a list of dicts with 'action_name' and 'params'.
    """
    if pd.isna(prediction_str):
        return []
    
    text_str = str(prediction_str)
    actions = []
    action_types = ['click', 'scroll', 'type', 'select', 'wait', 'finished']
    
    # Helper: Find balanced braces/parentheses
    def find_balanced(text, start_pos, open_char='(', close_char=')'):
        depth = 0
        for i in range(start_pos, len(text)):
            if text[i] == open_char:
                depth += 1
            elif text[i] == close_char:
                depth -= 1
                if depth == 0:
                    return i
        return -1
    
    # Helper: Parse coordinates from various formats
    def parse_coordinates(params_str):
        """Extract coordinates from params string. Returns dict with 'coords_2d', 'coords_4d', or None"""
        if not params_str:
            return None
        
        # Try to find numbers in the params
        numbers = re.findall(r'-?\d+\.?\d*', str(params_str))
        if not numbers:
            return None
        
        # Convert to floats/ints
        try:
            coords = [float(n) if '.' in n else int(n) for n in numbers]
        except ValueError:
            return None
        
        result = {}
        
        # Check for 2D coordinates (2 numbers)
        if len(coords) >= 2:
            result['coords_2d'] = coords[:2]
        
        # Check for 4D coordinates (4 numbers - bounding box)
        if len(coords) >= 4:
            result['coords_4d'] = coords[:4]
        
        # If we have more than 4 numbers, store them as 'coords_extra'
        if len(coords) > 4:
            result['coords_extra'] = coords
        
        return result if result else None
    
    # Helper: Extract params from JSON arguments based on action type
    def extract_json_params(action_name, args):
        """Extract relevant parameter from JSON arguments"""
        import json
        param_map = {
            'scroll': ['pixels', 'direction'],
            'type': ['text', 'content'],
            'click': ['start_box', 'coordinates']
        }
        for key in param_map.get(action_name, []):
            if key in args:
                return str(args[key])
        # Fallback: return all args except 'action'
        params = {k: v for k, v in args.items() if k != 'action'}
        return json.dumps(params, separators=(',', ':')) if params else None
    
    # Helper: Parse JSON and extract action
    def parse_json_action(json_str):
        """Parse JSON and return action dict if valid"""
        try:
            import json
            data = json.loads(json_str)
            if not isinstance(data, dict):
                return None
            
            # Find action name
            action_name = None
            if 'name' in data and data['name'] in action_types:
                action_name = data['name'].lower()
            elif 'arguments' in data and isinstance(data['arguments'], dict):
                if 'action' in data['arguments'] and data['arguments']['action'] in action_types:
                    action_name = data['arguments']['action'].lower()
            
            if not action_name:
                return None
            
            # Extract params
            params = None
            if 'arguments' in data and isinstance(data['arguments'], dict):
                params = extract_json_params(action_name, data['arguments'])
            
            action_dict = {
                'action_name': action_name,
                'params': params,
                'raw': json_str,
                'malformed': True
            }
            
            # Parse coordinates from params
            coords = parse_coordinates(params)
            if coords:
                action_dict.update(coords)
            
            return action_dict
        except (json.JSONDecodeError, ValueError):
            return None
    
    # Strategy 1: Well-formed actions after "Action:"
    action_section_match = re.search(r'Action:\s*(.+?)(?:\n|$)', text_str, re.IGNORECASE | re.DOTALL)
    if action_section_match:
        action_section = action_section_match.group(1).strip()
        for action_type in action_types:
            for match in re.finditer(rf'\b{action_type}\s*\(', action_section, re.IGNORECASE):
                start_pos = match.end() - 1
                end_pos = find_balanced(action_section, start_pos)
                if end_pos != -1:
                    params = action_section[match.end():end_pos].strip()
                    action_dict = {
                        'action_name': action_type.lower(),
                        'params': params,
                        'raw': action_section[match.start():end_pos + 1]
                    }
                    # Parse coordinates from params
                    coords = parse_coordinates(params)
                    if coords:
                        action_dict.update(coords)
                    actions.append(action_dict)
    
    # Strategy 2: JSON in tool_call tags or standalone JSON
    if len(actions) == 0:
        # Helper: Find all JSON objects starting at a position
        def find_all_json(text, start_pos=0):
            json_objects = []
            i = start_pos
            while i < len(text):
                if text[i] == '{':
                    end_pos = find_balanced(text, i, '{', '}')
                    if end_pos != -1:
                        json_objects.append(text[i:end_pos + 1])
                        i = end_pos + 1
                    else:
                        i += 1
                else:
                    i += 1
            return json_objects
        
        # Try tool_call tags first - find ALL JSON objects in tool_call sections
        tool_call_matches = list(re.finditer(r'<tool_call>', text_str, re.IGNORECASE))
        for tool_call_match in tool_call_matches:
            json_objects = find_all_json(text_str, tool_call_match.end())
            for json_str in json_objects:
                action = parse_json_action(json_str)
                if action:
                    actions.append(action)
        
        # Try finding any JSON with "action" field (if no tool_call found)
        if len(actions) == 0:
            json_objects = find_all_json(text_str)
            for json_str in json_objects:
                if '"action"' in json_str or "'action'" in json_str:
                    action = parse_json_action(json_str)
                    if action:
                        actions.append(action)
    
    # Strategy 3: Fallback - find action keywords, but EXCLUDE "Thought:" sections
    if len(actions) == 0:
        # Remove "Thought:" sections to avoid false matches
        # Find all "Thought:" sections and remove them
        search_text = text_str
        thought_pattern = r'Thought:\s*[^\n]*(?:\n|$)'
        search_text = re.sub(thought_pattern, '', search_text, flags=re.IGNORECASE | re.MULTILINE)
        
        # Search for action keywords in the filtered text (excluding Thought sections)
        for action_type in action_types:
            match = re.search(rf'\b{action_type}\b', search_text, re.IGNORECASE)
            if match:
                # Try to find coordinates near the action keyword (within 100 chars)
                start = max(0, match.start() - 50)
                end = min(len(search_text), match.end() + 50)
                context = search_text[start:end]
                
                action_dict = {
                    'action_name': action_type.lower(),
                    'params': None,
                    'raw': action_type,
                    'malformed': True
                }
                
                # Parse coordinates from context around the action
                coords = parse_coordinates(context)
                if coords:
                    action_dict.update(coords)
                
                actions.append(action_dict)
    
    return actions

# Test the function and apply to dataframe
df['parsed_actions'] = df['prediction'].apply(parse_actions)

# Also create a column with just action names for easier analysis
df['action_names'] = df['parsed_actions'].apply(lambda x: [a['action_name'] for a in x] if x else [])
df['action_count'] = df['parsed_actions'].apply(len)

# Extract coordinates from parsed actions for easier metric calculation
# Take coordinates from the first action (if any)
def extract_coords_from_actions(actions):
    """Extract coordinates from the first action"""
    if not actions or len(actions) == 0:
        return {
            'coords_2d': None,
            'coords_4d': None,
            'coords_extra': None
        }
    
    first_action = actions[0]
    return {
        'coords_2d': first_action.get('coords_2d'),
        'coords_4d': first_action.get('coords_4d'),
        'coords_extra': first_action.get('coords_extra')
    }

coords_df = pd.DataFrame(df['parsed_actions'].apply(extract_coords_from_actions).tolist())
df['coords_2d'] = coords_df['coords_2d']
df['coords_4d'] = coords_df['coords_4d']
df['coords_extra'] = coords_df['coords_extra']

# show full width
pd.set_option('display.max_colwidth', None)

# Filter for rows with 0 actions (malformed/missing) OR more than 1 action (multiple actions)
more_than_1_action = (df['prediction'].str.contains('tool_call')) | (df['action_count'] > 1) | (df['action_count'] == 0)

print(f"Sample parsed actions:")
print(f"Rows with 0 or more than 1 action: {more_than_1_action.sum()}")
print(f"Rows with exactly 1 action: {(df['action_count'] == 1).sum()}")
print(f"\nRows with coords_2d: {df['coords_2d'].notna().sum()}")
print(f"Rows with coords_4d: {df['coords_4d'].notna().sum()}")
display(df[more_than_1_action][['model','instruction', 'prediction', 'parsed_actions', 'coords_2d', 'coords_4d']].head())

Sample parsed actions:
Rows with 0 or more than 1 action: 21
Rows with exactly 1 action: 11817

Rows with coords_2d: 11590
Rows with coords_4d: 1257


,model,instruction,prediction,parsed_actions,coords_2d,coords_4d
230,qwen,Type 'Home Decor Inspiration' above 'Keep this board secret',"<tool_call>\n{""name"": ""computer_use"", ""arguments"": {""action"": ""scroll"", ""pixels"": -505}}\n</tool_call>","[{'action_name': 'scroll', 'params': '-505', 'raw': '{""name"": ""computer_use"", ""arguments"": {""action"": ""scroll"", ""pixels"": -505}}', 'malformed': True}]",None,None
1055,qwen,Type 'Home Decor Inspiration' above 'Keep this board secret',"<tool_call>\n{""name"": ""computer_use"", ""arguments"": {""action"": ""scroll"", ""pixels"": -505}}\n</tool_call>","[{'action_name': 'scroll', 'params': '-505', 'raw': '{""name"": ""computer_use"", ""arguments"": {""action"": ""scroll"", ""pixels"": -505}}', 'malformed': True}]",None,None
1250,qwen,Click on the button above 'Done',"Thought: The goal is to click on the button labeled ""Done"" which is likely located above the current view. Since it's not visible in the screenshot, I will scroll down to locate it.\nAction: scroll(start_box='Box(0,0)'), wait()","[{'action_name': 'scroll', 'params': 'start_box='Box(0,0)'', 'raw': 'scroll(start_box='Box(0,0)')', 'coords_2d': [0, 0]}, {'action_name': 'wait', 'params': '', 'raw': 'wait()'}]","[0, 0]",None
1572,qwen,Click on the text below 'Cheapest',"Thought: The task is to click on the text below 'Cheapest'. I need to locate the text and click on it. The text is likely to be near the bottom of the screen, so I will scroll down slightly if necessary before clicking.\n\nAction: scroll(start_box='Box(498,602,1931,1087)'), wait()","[{'action_name': 'scroll', 'params': 'start_box='Box(498,602,1931,1087)'', 'raw': 'scroll(start_box='Box(498,602,1931,1087)')', 'coords_2d': [498, 602], 'coords_4d': [498, 602, 1931, 1087]}, {'action_name': 'wait', 'params': '', 'raw': 'wait()'}]","[498, 602]","[498, 602, 1931, 1087]"
1634,qwen,Click on the link above 'Italy',"Thought: The goal is to click on the link labeled 'Italy' which is located above the current view. This will likely lead to a page specific to vinyl records from Italy. The link is not visible in the current screenshot, so I need to scroll down to find it.\n\nAction: scroll(start_box='Box(0,0)'), wait()","[{'action_name': 'scroll', 'params': 'start_box='Box(0,0)'', 'raw': 'scroll(start_box='Box(0,0)')', 'coords_2d': [0, 0]}, {'action_name': 'wait', 'params': '', 'raw': 'wait()'}]","[0, 0]",None


In [11]:
contains_type_action = df['prediction'].str.contains('type')
print(f"Rows with type action: {contains_type_action.sum()}")
# display(df[contains_type_action][['model','instruction', 'prediction', 'parsed_actions', 'action_names', 'action_count', 'coords_2d', 'coords_4d']].head(-10))
display(df[contains_type_action][['model', 'reasoning_type', 'parsed_actions', 'prediction_type', 'action_names']].groupby(['model', 'reasoning_type', 'prediction_type']).head(5))


Rows with type action: 304


,model,reasoning_type,parsed_actions,prediction_type,action_names
1187,qwen,reasoning,"[{'action_name': 'click', 'params': 'start_box='_bbox(449,546),(832,583)_'', 'raw': 'click(start_box='_bbox(449,546),(832,583)_')', 'coords_2d': [449, 546], 'coords_4d': [449, 546, 832, 583]}]",has_thought + 4_numbers + click,[click]
1197,qwen,reasoning,"[{'action_name': 'click', 'params': 'start_box='_bbox(1029,430,1491,482)_bbox'', 'raw': 'click(start_box='_bbox(1029,430,1491,482)_bbox')', 'coords_2d': [1029, 430], 'coords_4d': [1029, 430, 1491, 482]}]",has_thought + 4_numbers + click,[click]
1216,qwen,reasoning,"[{'action_name': 'click', 'params': 'start_box='_bbox(136,229),(478,407)_'', 'raw': 'click(start_box='_bbox(136,229),(478,407)_')', 'coords_2d': [136, 229], 'coords_4d': [136, 229, 478, 407]}]",has_thought + 4_numbers + click,[click]
1232,qwen,reasoning,"[{'action_name': 'click', 'params': 'start_box='(602,47)'', 'raw': 'click(start_box='(602,47)')', 'coords_2d': [602, 47]}]",has_thought + click,[click]
1246,qwen,reasoning,"[{'action_name': 'click', 'params': 'start_box='[1204,121]'', 'raw': 'click(start_box='[1204,121]')', 'coords_2d': [1204, 121]}]",has_thought + click,[click]
1260,qwen,reasoning,"[{'action_name': 'type', 'params': 'content='42701\n'', 'raw': 'type(content='42701\n')'}]",has_thought + type,[type]
1262,qwen,reasoning,"[{'action_name': 'click', 'params': 'start_box='_bbox(847,70,912,106)'', 'raw': 'click(start_box='_bbox(847,70,912,106)')', 'coords_2d': [847, 70], 'coords_4d': [847, 70, 912, 106]}]",has_thought + 4_numbers + click,[click]
1275,qwen,reasoning,"[{'action_name': 'click', 'params': 'start_box='(480,130)'', 'raw': 'click(start_box='(480,130)')', 'coords_2d': [480, 130]}]",has_thought + 4_numbers + click,[click]
1305,qwen,reasoning,"[{'action_name': 'click', 'params': 'start_box='[1142,160]'', 'raw': 'click(start_box='[1142,160]')', 'coords_2d': [1142, 160]}]",has_thought + click,[click]
1311,qwen,reasoning,"[{'action_name': 'click', 'params': 'start_box='_bbox(867,402,911,426)_'', 'raw': 'click(start_box='_bbox(867,402,911,426)_')', 'coords_2d': [867, 402], 'coords_4d': [867, 402, 911, 426]}]",has_thought + click,[click]


In [12]:
has_type = (df['prediction'].str.contains('type')) & (df['action_names'].apply(lambda x: 'type' in x))
df[has_type][['model', 'reasoning_type', 'prediction_type', 'prediction']].groupby(['model', 'reasoning_type', 'prediction_type']).value_counts()

model   reasoning_type  prediction_type                  prediction                                                                                                                                                                                                                                                                                                                                        
qwen    no_reasoning    no_thought + contains_tool_call  <tool_call>\n{"name": "computer_use", "arguments": {"action": "type", "text": "PUNE"}}\n</tool_call>                                                                                                                                                                                                                                                  1
        reasoning       has_thought + type               Thought: The goal is to type 'diabetes' into the search box. The search box is clearly visible and ready for input.\nAction: type(content='diabete

# Fixing coordinates by adding correct coordinate denormalization using W_RATIO & H_RATIO & converting 4D predictions to 2D predictions

In [13]:
SMART_RESIZE = (1932, 1092)
IMAGE_SIZE = (1920, 1080)

W_RATIO = IMAGE_SIZE[0] / SMART_RESIZE[0]
H_RATIO = IMAGE_SIZE[1] / SMART_RESIZE[1]

In [14]:
df.head()

,model,reasoning_type,query_type,test_split,variant,task_id,step_index,instruction,gt_bbox,prediction,action_str_em,hit_box_accuracy,bbox_center_mse,prediction_type,parsed_actions,action_names,action_count,coords_2d,coords_4d,coords_extra
0,qwen,no_reasoning,relational_query,task,text_shrink,8dcf6423-262a-439b-9ee7-279a920468fa,0,Click on the button to the left of 'EXPERIENCE',"[370, 61, 111.265625, 55]","Action: click(start_box='(553,86)')",1.0,0.0,8114.325226,no_thought + click,"[{'action_name': 'click', 'params': 'start_box='(553,86)'', 'raw': 'click(start_box='(553,86)')', 'coords_2d': [553, 86]}]",[click],1,"[553, 86]",None,None
1,qwen,no_reasoning,relational_query,task,text_shrink,e6f6e6c8-f1e6-42bb-a3af-696ed8de571b,0,Click on the link above 'Flight status',"[607.046875, 174.390625, 65.390625, 12]","Action: click(start_box='(640,211)')",1.0,0.0,468.500153,no_thought + click,"[{'action_name': 'click', 'params': 'start_box='(640,211)'', 'raw': 'click(start_box='(640,211)')', 'coords_2d': [640, 211]}]",[click],1,"[640, 211]",None,None
2,qwen,no_reasoning,relational_query,task,text_shrink,400c291f-6a0c-46fb-874e-d5c174fdedfc,0,Click on the button below 'Infant +' button,"[895.5, 533.5, 32.5, 13]","Action: click(start_box='(834,500)')",1.0,0.0,3822.531250,no_thought + click,"[{'action_name': 'click', 'params': 'start_box='(834,500)'', 'raw': 'click(start_box='(834,500)')', 'coords_2d': [834, 500]}]",[click],1,"[834, 500]",None,None
3,qwen,no_reasoning,relational_query,task,text_shrink,5b888855-b921-4c61-8f79-73902ee0eafa,0,Type the textbox above '* PIN Number',"[448, 461.953125, 380, 36.890625]","Action: click(start_box='(641,563)')",0.0,0.0,3416.009064,no_thought + click,"[{'action_name': 'click', 'params': 'start_box='(641,563)'', 'raw': 'click(start_box='(641,563)')', 'coords_2d': [641, 563]}]",[click],1,"[641, 563]",None,None
4,qwen,no_reasoning,relational_query,task,text_shrink,9ceab2a3-7919-4f15-871a-21638fd93b24,0,"Select '11:00 AM' in the select to the left of 'Drop-off date Mon, Apr 3'","[1036.40625, 202, 112, 62]","Action: click(start_box='(1356,247)')",0.0,0.0,34838.832520,no_thought + click,"[{'action_name': 'click', 'params': 'start_box='(1356,247)'', 'raw': 'click(start_box='(1356,247)')', 'coords_2d': [1356, 247]}]",[click],1,"[1356, 247]",None,None


In [15]:
"""
1. coords_4d exists -> calculate center -> corrected_coords_2d
2. coords_4d doesn't exist -> coords_2d is corrected_coords_2d
3. coords_extra exists -> does step 1
4. coords_2d is None likely due to Type action -> save None coords_2d as corrected_coords_2d
"""

def get_center_from_qwen_coords_4d(coords_4d):
    x1, y1, x2, y2 = coords_4d
    return [(x1 + x2) / 2, (y1 + y2) / 2]

# get center from coords_4d for qwen rows with 4d coordinates
df['corrected_coords_2d'] = df.apply(lambda x: get_center_from_qwen_coords_4d(x['coords_4d']) if x['coords_4d'] is not None else x['coords_2d'], axis=1)

# denormalize coords_2d for all models
df['corrected_coords_2d_denormalized'] = df.apply(lambda x: [int(x['corrected_coords_2d'][0] * W_RATIO), int(x['corrected_coords_2d'][1] * H_RATIO)] if x['corrected_coords_2d'] is not None else x['corrected_coords_2d'], axis=1)


In [16]:
df[df['coords_4d'].notna()][['model', 'reasoning_type', 'coords_2d', 'coords_4d', 'corrected_coords_2d', 'corrected_coords_2d_denormalized']].head()

,model,reasoning_type,coords_2d,coords_4d,corrected_coords_2d,corrected_coords_2d_denormalized
1184,qwen,reasoning,"[524, 80]","[524, 80, 597, 93]","[560.5, 86.5]","[557, 85]"
1186,qwen,reasoning,"[620, 484]","[620, 484, 679, 502]","[649.5, 493.0]","[645, 487]"
1187,qwen,reasoning,"[449, 546]","[449, 546, 832, 583]","[640.5, 564.5]","[636, 558]"
1188,qwen,reasoning,"[1297, 204]","[1297, 204, 1414, 268]","[1355.5, 236.0]","[1347, 233]"
1190,qwen,reasoning,"[568, 409]","[568, 409, 1247, 481]","[907.5, 445.0]","[901, 440]"


# Fix the action str exact match, hit box accuracy, & bbox center mse, and add normalized mse by bbox diagonal length

## Calculate action str exact match

1. malformed -> 0
2. non-coordinate actions -> action type and params must match
3. normal click action with coordinates case, see if action type matches
4. for steps with 2 actions, it's only scroll, wait. and we take the first action 'scroll'

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11835 entries, 0 to 11834
Data columns (total 22 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   model                             11835 non-null  object 
 1   reasoning_type                    11835 non-null  object 
 2   query_type                        11835 non-null  object 
 3   test_split                        11835 non-null  object 
 4   variant                           11835 non-null  object 
 5   task_id                           11835 non-null  object 
 6   step_index                        11835 non-null  int64  
 7   instruction                       11835 non-null  object 
 8   gt_bbox                           11835 non-null  object 
 9   prediction                        11835 non-null  object 
 10  action_str_em                     10264 non-null  float64
 11  hit_box_accuracy                  10026 non-null  float64
 12  bbox

In [ ]:
# We are only counting scroll as mismatch
# Type
# Click
# Select
# Scroll
# Wait
# Finished

# df.info()
# df.head()
# df[df['action_count'] != 1][['model', 'prediction_type']].value_counts()
# df[df[('action_count' != 1) & (df['action_names'][1] != 'wait')]][['model', 'action_names']].value_counts()
# df[(df['action_count'] != 1) & (~df['action_names'].apply(lambda x: 'wait' in x))][['model', 'action_names']].head()


def get_action_str_em(row):
    gt_action_type = row['gt']
    action_name = row['parsed_actions'][0]['action_name']
    if action_name == 'type':
        return 0
    elif action_name == 'click':
        return 1
    else:
        return 0



,model,action_names
1250,qwen,"[scroll, wait]"
1572,qwen,"[scroll, wait]"
1634,qwen,"[scroll, wait]"
1660,qwen,"[scroll, wait]"
1722,qwen,"[scroll, wait]"


## Calculate hit box accuracy

1. falls inside bbox -> 1
2. malformed -> 0
3. non-coordinate actions -> 0 ?? based on our parsing logic, model didn't output any coordinates? Check type action parsing and mapping 

In [ ]:
# print(df['coords_2d'].notna().sum())
# print(df['coords_4d'].notna().sum())
# print(df['coords_extra'].notna().sum())

# Convert coords_4d to tuple for grouping (lists are not hashable)
df_temp = df[df['coords_4d'].notna()].copy()
df_temp['coords_4d_str'] = df_temp['coords_4d'].apply(lambda x: tuple(x) if x is not None else None)
# display(df_temp.groupby(['model', 'reasoning_type', 'prediction', 'coords_4d_str'])['prediction'].count())

def is_coords_in_bbox(coords, bbox):
    x1, y1, w, h = bbox
    x, y = coords
    return x1 <= x <= x1 + w and y1 <= y <= y1 + h

# 2d is_coords_in_bbox(coords_2d, gt_bbox)

# extra coordinates, just skip and use coords_4d

# 4d get center from the 4d coordinates
# For qwen, coords_4d means (x1, y1, x2, y2) and get center x = (x1 + x2) / 2, center y = (y1 + y2) / 2


print(len(df[(df['coords_4d'].notna()) & (df['model'] == 'qwen')]))
print(len(df[(df['coords_4d'].notna()) & (df['model'] == 'gta1')]))
print(len(df[(df['coords_4d'].notna()) & (df['model'] == 'uitars')]))




11590
1257
1
1257
0
0


## Calculate bbox center mse & normalized mse

1. 2D coordinates -> MSE / bbox diagonal length^2:
    $NMSE_i​=\frac{​(x_c​−\hat{x})^2+(y_c​−\hat{y}​)^2​}{D^2_{gt}}$
2. non-coordinate actions -> max possible squared error

In [ ]:
"""
1. get nmse for normal coordinates

For None coords_2d
2. calculate max mse & nmse from the rows with normal coordinates
3. use 95% quantile of max mse and max nmse, as the threshold for NMSE_MALFORM_PENALTY
"""


def get_bbox_normalized_mse(pred, gt_bbox):
    """
    Calculate the normalized mean squared error of the bbox center

    Args:
        pred: (x, y)
        gt_bbox: (x1, y1, w, h)
    Returns:
        NMSE
    """
    x1, y1, w, h = gt_bbox
    diagonal_squared = w ** 2 + h ** 2

    if pred is None:
        return None, None  # calculate later

    x, y = pred
    gt_center = (x1 + w / 2, y1 + h / 2) # gt_bbox follows the format of (x1, y1, w, h)
    mse = ((x - gt_center[0]) ** 2 + (y - gt_center[1]) ** 2)
    
    return mse, mse / diagonal_squared

df['bbox_center_mse'], df['normalized_mse'] = df.apply(lambda x: get_bbox_normalized_mse(x['coords_2d'], x['gt_bbox']), axis=1)


# get max mse & nmse from the rows with normal coordinates

# set NMSE_MALFORM_PENALTY to 95% quantile of max mse and max nmse

# set None mse and nmse to NMSE_MALFORM_PENALTY


# Save df as csv

In [ ]:
df.to_csv('baseline_results_final.csv', index=False)